In [2]:
import os
import cv2
import random
import shutil
import pathlib
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [3]:
CLASS_NAMES = {
    0: "Caries",
    6: "Missing_Teeth",
    7: "Periapical_Lesion",
    11:"Impacted_Tooth",
    13:"Bone_Loss"
}

In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

In [5]:
YOLO_ROOT = pathlib.Path(r"D:\FYP\dental-vision\ML\data\dentaldataset01\YOLO")

ALL_CROPS = pathlib.Path(r"D:\FYP\dental-vision\ML\data\all_crops")

OUTPUT = pathlib.Path(r"D:\FYP\dental-vision\ML\data\processed")

In [6]:
if ALL_CROPS.exists():
    shutil.rmtree(ALL_CROPS)

ALL_CROPS.mkdir(parents=True)

for disease in CLASS_NAMES.values():
    (ALL_CROPS / disease).mkdir(parents=True, exist_ok=True)

print("Folders created successfully.")

Folders created successfully.


In [8]:
def crop_dataset(split):

    image_dir = YOLO_ROOT / split / "images"
    label_dir = YOLO_ROOT / split / "labels"

    total = 0

    for label_file in tqdm(sorted(label_dir.glob("*.txt"))):

        image_path = None

        for ext in [".jpg",".jpeg",".png",".JPG",".PNG",".JPEG"]:
            candidate = image_dir / (label_file.stem + ext)
            if candidate.exists():
                image_path = candidate
                break

        if image_path is None:
            continue

        image = cv2.imread(str(image_path))

        if image is None:
            continue

        h,w,_ = image.shape

        with open(label_file) as f:

            index = 0

            for line in f:

                values = line.strip().split()

                if len(values)==0:
                    continue

                cls = int(values[0])

                if cls not in CLASS_NAMES:
                    continue

                x,y,bw,bh = map(float, values[1:5])

                xc = x*w
                yc = y*h

                bw *= w
                bh *= h

                pad_x = bw*0.15
                pad_y = bh*0.15

                xmin = int(max(0, xc-bw/2-pad_x))
                xmax = int(min(w, xc+bw/2+pad_x))

                ymin = int(max(0, yc-bh/2-pad_y))
                ymax = int(min(h, yc+bh/2+pad_y))

                crop = image[ymin:ymax, xmin:xmax]

                save_folder = ALL_CROPS / CLASS_NAMES[cls]

                filename = f"{image_path.stem}_{split}_{index}.jpg"

                cv2.imwrite(str(save_folder / filename), crop)

                total += 1
                index += 1

    print(f"{split} : {total} crops")

In [9]:
for split in ["train","valid","test"]:
    crop_dataset(split)

100%|██████████| 9331/9331 [03:40<00:00, 42.38it/s]


train : 34238 crops


100%|██████████| 2871/2871 [01:14<00:00, 38.46it/s]


valid : 10196 crops


100%|██████████| 1730/1730 [00:47<00:00, 36.14it/s]

test : 6194 crops


In [10]:
print(f"{'Disease':25s}Images")

print("-"*40)

for disease in CLASS_NAMES.values():

    total = len(list((ALL_CROPS/disease).glob("*.jpg")))

    print(f"{disease:25s}{total}")

Disease                  Images
----------------------------------------
Caries                   10724
Missing_Teeth            3505
Periapical_Lesion        5291
Impacted_Tooth           27978
Bone_Loss                3130


In [11]:
if OUTPUT.exists():
    shutil.rmtree(OUTPUT)

for split in ["train","valid","test"]:

    for disease in CLASS_NAMES.values():

        (OUTPUT/split/disease).mkdir(parents=True,exist_ok=True)

In [12]:
def split_class(class_name):

    images = list((ALL_CROPS/class_name).glob("*.jpg"))

    train, temp = train_test_split(
        images,
        test_size=0.30,
        random_state=SEED,
        shuffle=True
    )

    valid, test = train_test_split(
        temp,
        test_size=0.50,
        random_state=SEED
    )

    return train, valid, test

In [13]:
for disease in CLASS_NAMES.values():

    train, valid, test = split_class(disease)

    for img in train:
        shutil.copy(img, OUTPUT/"train"/disease/img.name)

    for img in valid:
        shutil.copy(img, OUTPUT/"valid"/disease/img.name)

    for img in test:
        shutil.copy(img, OUTPUT/"test"/disease/img.name)

print("Dataset split completed.")

Dataset split completed.


In [14]:
print(f"{'Disease':25s}{'Train':>8}{'Valid':>8}{'Test':>8}")

print("-"*50)

for disease in CLASS_NAMES.values():

    train = len(list((OUTPUT/"train"/disease).glob("*.jpg")))
    valid = len(list((OUTPUT/"valid"/disease).glob("*.jpg")))
    test = len(list((OUTPUT/"test"/disease).glob("*.jpg")))

    print(f"{disease:25s}{train:8d}{valid:8d}{test:8d}")

Disease                     Train   Valid    Test
--------------------------------------------------
Caries                       7506    1609    1609
Missing_Teeth                2453     526     526
Periapical_Lesion            3703     794     794
Impacted_Tooth              19584    4197    4197
Bone_Loss                    2191     469     470


In [15]:
for disease in CLASS_NAMES.values():

    train = len(list((OUTPUT/"train"/disease).glob("*.jpg")))
    valid = len(list((OUTPUT/"valid"/disease).glob("*.jpg")))
    test = len(list((OUTPUT/"test"/disease).glob("*.jpg")))

    total = train + valid + test

    print(f"{disease} : {total}")

Caries : 10724
Missing_Teeth : 3505
Periapical_Lesion : 5291
Impacted_Tooth : 27978
Bone_Loss : 3130


In [16]:
train_files = set()
valid_files = set()
test_files = set()

for disease in CLASS_NAMES.values():

    train_files.update([img.name for img in (OUTPUT/"train"/disease).glob("*.jpg")])

    valid_files.update([img.name for img in (OUTPUT/"valid"/disease).glob("*.jpg")])

    test_files.update([img.name for img in (OUTPUT/"test"/disease).glob("*.jpg")])

print("Train ∩ Valid :", len(train_files & valid_files))
print("Train ∩ Test  :", len(train_files & test_files))
print("Valid ∩ Test  :", len(valid_files & test_files))

Train ∩ Valid : 0
Train ∩ Test  : 0
Valid ∩ Test  : 0


In [17]:
from collections import Counter
import cv2

sizes = Counter()

for disease in CLASS_NAMES.values():
    for img_path in (OUTPUT/"train"/disease).glob("*.jpg"):
        img = cv2.imread(str(img_path))
        if img is not None:
            h, w = img.shape[:2]
            sizes[(w, h)] += 1

print("Unique image sizes:", len(sizes))
print(sizes.most_common(10))

Unique image sizes: 26591
[((491, 491), 8), ((494, 496), 8), ((495, 500), 8), ((444, 508), 8), ((464, 495), 7), ((439, 497), 7), ((503, 477), 7), ((485, 496), 7), ((435, 490), 7), ((509, 467), 7)]
